# 02 — RMSD クラスタリング / RMSD Clustering

`01_processing.ipynb` で生成した SDF ファイルから全ポーズを読み込み、
RMSD 行列を計算して階層クラスタリングを実行します。
結果を `all_poses_cluster.sdf` と `all_poses.csv` に保存します。

Loads all poses from the SDF files produced by `01_processing.ipynb`,
computes an RMSD matrix, performs hierarchical clustering, and saves
`all_poses_cluster.sdf` and `all_poses.csv`.

**以下の CONFIG セルのみ編集してください。 / Edit only the CONFIG cell below.**

In [ ]:
# CONFIG -----------------------------------------------------------------------
CONFIG_PATH = "../notebooks/templates/project_config.toml"  # ← edit this path
# ------------------------------------------------------------------------------

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import PandasTools

from docking_analysis import (
    AnalysisConfig,
    compute_rmsd_matrix,
    cluster_poses,
    find_optimal_threshold,
    plot_dendrogram,
)

from docking_analysis.visualization.contact_heatmap import (
    build_contact_rate_matrix,
    plot_contact_heatmap,
)

In [ ]:
config = AnalysisConfig.from_toml(CONFIG_PATH)
cfg_c  = config.clustering
print(f"Project        : {config.project_name}")
print(f"Target clusters: {cfg_c.target_clusters}")
print(f"Linkage method : {cfg_c.method}")

## 全ポーズの読み込み / Load all poses

In [ ]:
sdf_files = sorted(config.results_dir.glob("*.sdf"))
# Exclude output files from a previous run of this notebook
sdf_files = [f for f in sdf_files if "all_poses" not in f.stem]

all_mols = []
for f in sdf_files:
    mols = [m for m in Chem.SDMolSupplier(str(f)) if m is not None]
    all_mols.extend(mols)
    print(f"  {f.name}: {len(mols)} poses")

print(f"\nTotal poses: {len(all_mols)}")

## 最適 RMSD 閾値の探索 / Find optimal RMSD threshold

In [ ]:
print("Pre-computing RMSD matrix …")
rmsd_matrix = compute_rmsd_matrix(all_mols, align=False)
print(f"RMSD matrix shape: {rmsd_matrix.shape}")
print(f"RMSD range: {rmsd_matrix.min():.2f} – {rmsd_matrix.max():.2f} Å")

In [ ]:
# Quick pre-cluster to get the linkage matrix
pre_result = cluster_poses(all_mols, rmsd_threshold=1.0, method=cfg_c.method)

optimal_threshold = find_optimal_threshold(
    pre_result.linkage_matrix,
    target_clusters=cfg_c.target_clusters,
    search_range=cfg_c.threshold_search_range,
    step=cfg_c.threshold_step,
)
print(f"Optimal threshold: {optimal_threshold:.1f} Å")

## 最適閾値でクラスタリング / Cluster with optimal threshold

In [ ]:
result = cluster_poses(all_mols, rmsd_threshold=optimal_threshold, method=cfg_c.method)
print(f"Number of clusters: {result.n_clusters}")

In [ ]:
fig = plot_dendrogram(
    result.linkage_matrix,
    threshold=optimal_threshold,
    output_path=config.results_dir / "dendrogram.png",
)
plt.show()

## クラスター ID を統合 DataFrame にマージして保存
## Merge cluster IDs into unified DataFrame and save

In [ ]:
# Save clustered SDF
clustered_sdf = config.results_dir / "all_poses_cluster.sdf"
writer = Chem.SDWriter(str(clustered_sdf))
for mol in result.mols:
    writer.write(mol)
writer.close()
print(f"Saved: {clustered_sdf}")

# Load all CSVs and concatenate
csv_files = sorted(config.results_dir.glob("*.csv"))
csv_files = [f for f in csv_files if "all_poses" not in f.stem]
all_df = pd.concat(
    [pd.read_csv(f) for f in csv_files], ignore_index=True
).fillna(0)

# Merge cluster IDs
mol_df = PandasTools.LoadSDF(str(clustered_sdf))
all_df = all_df.merge(mol_df[["mol_name", "cluster_id"]], on="mol_name")
all_df["cluster_id"] = all_df["cluster_id"].astype(int)

all_csv = config.results_dir / "all_poses.csv"
all_df.to_csv(all_csv, index=False)
print(f"Saved: {all_csv}  ({all_df.shape[0]} poses × {all_df.shape[1]} cols)")

## スコア分布の概観 / Score distribution overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

all_df["docking_score"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_xlabel("Docking score (kcal/mol)")
axes[0].set_ylabel("Count")
axes[0].set_title("Score distribution — all poses")

top_per_cluster = all_df.sort_values("docking_score").drop_duplicates("cluster_id")
top_per_cluster["docking_score"].hist(bins=20, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_xlabel("Docking score (kcal/mol)")
axes[1].set_title("Score distribution — cluster representatives")

plt.tight_layout()
plt.show()

## 残基接触ヒートマップ / Residue Contact Heatmap

ProLIF インタラクション列を使って、クラスターごとの残基接触率をヒートマップで可視化します。

Visualise per-cluster residue contact rates using ProLIF interaction columns.

In [ ]:
# ProLIF 列（バイナリ接触フラグ）を自動検出
# Auto-detect ProLIF binary interaction columns
residue_cols = [
    c for c in all_df.columns
    if any(c.endswith(t) for t in ["HBAcceptor","HBDonor","Hydrophobic","Anionic","Cationic","PiStacking","PiCation"])
]
print(f"Detected {len(residue_cols)} residue interaction columns")

if residue_cols:
    fig = plot_contact_heatmap(
        all_df,
        residue_cols,
        group_col="cluster_id",
        cluster_rows=True,       # 階層クラスタリングで行を並べ替え / reorder rows by dendrogram
        annotate=True,           # セル内に数値を表示 / annotate cells with values
        output_path=config.results_dir / "contact_heatmap.png",
    )
    plt.show()
    print(f"Saved: {config.results_dir / 'contact_heatmap.png'}")
else:
    print("No ProLIF interaction columns found — run 01_processing.ipynb first.")